In [1]:
# from database import get_connection, release_connection, create_tables
import psycopg2
import psycopg2.extras
import pandas as pd

In [99]:
connection = psycopg2.connect(database="mydb", user="myuser", password="mypassword", host="localhost", port=5433, options="-c client_encoding=UTF8")
connection.set_client_encoding('UTF8')

cursor = connection.cursor()

connection1 = psycopg2.connect(database="mydbtest", user="myuser", password="mypassword", host="localhost", port=5433, options="-c client_encoding=UTF8")
connection1.set_client_encoding('UTF8')

cursor1 = connection1.cursor()

In [ ]:
df = pd.read_csv('../../grammar_points.csv', names=["level", "grammar_point_jp", "meaning_en", "description", "url"])
df

desc = df["description"].iloc[0]
desc


['[', '[', 'N', 'o', 'n', 'e', ',', ' ', 'N', 'o', 'n', 'e', ',', ' ', "'", 'F', 'o', 'r', ' ', 't', 'h', 'e', ' ', 'm', 'o', 's', 't', ' ', 'p', 'a', 'r', 't', ',', ' ', "'", ',', ' ', 'N', 'o', 'n', 'e', ']', ',', ' ', '[', 'N', 'o', 'n', 'e', ',', ' ', 'N', 'o', 'n', 'e', ',', ' ', "'", 'だ', "'", ',', ' ', 'N', 'o', 'n', 'e', ']', ',', ' ', '[', 'N', 'o', 'n', 'e', ',', ' ', 'N', 'o', 'n', 'e', ',', ' ', "'", ' ', 'i', 's', ' ', 't', 'h', 'e', ' ', 'e', 'q', 'u', 'i', 'v', 'a', 'l', 'e', 'n', 't', ' ', 'o', 'f', ' ', '@', '@', 'i', 's', '@', '@', ' ', 'i', 'n', ' ', 'E', 'n', 'g', 'l', 'i', 's', 'h', '.', ' ', 'I', 't', 's', ' ', 'r', 'o', 'l', 'e', ' ', 'i', 's', ' ', 't', 'o', ' ', 's', 't', 'r', 'o', 'n', 'g', 'l', 'y', ' ', 'e', 'x', 'p', 'r', 'e', 's', 's', ' ', 'd', 'e', 't', 'e', 'r', 'm', 'i', 'n', 'a', 't', 'i', 'o', 'n', ' ', 'o', 'r', ' ', 'a', 's', 's', 'e', 'r', 't', 'i', 'o', 'n', '.', ' ', 'I', 't', ' ', 'i', 's', ' ', 'a', ' ', 'c', 'a', 's', 'u', 'a', 'l', ' ', 'g',

In [83]:
import numpy as np
import json

def stringToList(string: str):
    # Returns text metadata retrieved from database as a list
    # Format of text metadata is [[kanji: str, furigana: str, hiragana : str, grammar : Bool]]
    list = []
    i = 1
    start_of_quote = False
    word = ''
    while i < len(string)-1:

        char = string[i] 
        if not i == len(string)-1:
            next_char = string[i+1]
        # print(char)
        
        if char == '[':
            sub_list = []
        elif char == ']':
            list.append(sub_list)
        elif char == ',' and not start_of_quote:
            pass
        elif char == " " and not start_of_quote:
            pass
        elif char == " " and start_of_quote:
            word += char
        else:
            if not start_of_quote:
                if char == "N": # Always equals None
                    sub_list.append(None)
                    i += 3
                elif char == "T":
                    sub_list.append(True)
                    i += 3
                elif char == "F":
                    sub_list.append(False)
                    i += 4
            if char == "'":
                if not start_of_quote: # If char is the opening quotation mark
                    word = '' 
                else:
                    sub_list.append(word)
                start_of_quote = not start_of_quote
            else: # Any letters in a quote
                word += char

        i += 1

    return list

def cleanStrings(arr: list):
    for i in arr:
        if i[2] is not None:
            # print(i[2])
            i[2] = i[2].replace("@@", "'")
    return arr

        
def formatStringsHtml(arr: list):
    string = ''
    for i in arr:
        if i[0] is not None:
            string += f'<ruby>{i[0]}<rp>(</rp><rt>{i[1]}</rt><rp>)</rp></ruby>'
        else:
            string += f'{i[2]}'
    
    return string

In [100]:
cursor.execute("""SELECT id, level, grammar_jp, grammar_en, description, url FROM grammar WHERE id = %s;""", (54,))
results = cursor.fetchone()
# print(f"Results: {results['id']}")

(id, level, grammar_jp, grammar_en, description, url) = results[0], results[1], results[2], results[3], results[4], results[5]

print(description)
description = stringToList(description)
print(description)
description = cleanStrings(description)
print(description)
description = formatStringsHtml(description)
print(description)

[[None, None, 'から', None], [None, None, ' can have several different meanings in Japanese, depending on which part of the sentence it is in, and what comes before/after it. It is often translated as @@from@@. In these cases, it just means @@with (A) as a starting location, (B)@@. In this grammar construction, ', None], [None, None, 'から', None], [None, None, ' comes directly after the place that is considered the starting point.', None], [None, None, 'This form of ', None], [None, None, 'から', None], [None, None, ' is the closest to the @@base@@ meaning of the word in Japanese, as the nuance of ', None], [None, None, 'から', None], [None, None, ' can almost always be thought of as meaning @@from@@ in some way.', None], [None, None, 'Caution', None], [None, None, 'This form of ', None], [None, None, 'から', None], [None, None, ' does not require ', None], [None, None, 'だ', None], [None, None, ' when used after nouns or ', None], [None, None, 'な-Adjectives', None], [None, None, ', as that woul

In [97]:
cursor1.execute("""SELECT id, level, grammar_jp, grammar_en, description, url FROM grammar WHERE id = %s;""", (54,))
results = cursor1.fetchone()
# print(f"Results: {results['id']}")

(id, level, grammar_jp, grammar_en, description, url) = results[0], results[1], results[2], results[3], results[4], results[5]

print(description)
description = stringToList(description)
print(description)
description = cleanStrings(description)
print(description)
description = formatStringsHtml(description)
print(description)

[[None, None, 'から', None], [None, None, ' can have several different meanings in Japanese, depending on which part of the sentence it is in, and what comes before/after it. It is often translated as @@from@@. In these cases, it just means @@with (A) as a starting location, (B)@@. In this grammar construction, ', None], [None, None, 'から', None], [None, None, ' comes directly after the place that is considered the starting point.', None], [None, None, 'This form of ', None], [None, None, 'から', None], [None, None, ' is the closest to the @@base@@ meaning of the word in Japanese, as the nuance of ', None], [None, None, 'から', None], [None, None, ' can almost always be thought of as meaning @@from@@ in some way.', None], [None, None, 'Caution', None], [None, None, 'This form of ', None], [None, None, 'から', None], [None, None, ' does not require ', None], [None, None, 'だ', None], [None, None, ' when used after nouns or ', None], [None, None, 'な-Adjectives', None], [None, None, ', as that woul

In [73]:
cursor.execute("SELECT * FROM grammar WHERE id = 54;")

results = cursor.fetchone()

print(results)
description = results[4]

description = stringToList(description)
print(description)
description = cleanStrings(description)
print(description)

description = formatStringsHtml(description)
print(description)

(54, 5, "[[None, None, 'から', None]]", 'From', "[[None, None, 'から', None], [None, None, ' can have several different meanings in Japanese, depending on which part of the sentence it is in, and what comes before/after it. It is often translated as @@from@@. In these cases, it just means @@with (A) as a starting location, (B)@@. In this grammar construction, ', None], [None, None, 'から', None], [None, None, ' comes directly after the place that is considered the starting point.', None], [None, None, 'This form of ', None], [None, None, 'から', None], [None, None, ' is the closest to the @@base@@ meaning of the word in Japanese, as the nuance of ', None], [None, None, 'から', None], [None, None, ' can almost always be thought of as meaning @@from@@ in some way.', None], [None, None, 'Caution', None], [None, None, 'This form of ', None], [None, None, 'から', None], [None, None, ' does not require ', None], [None, None, 'だ', None], [None, None, ' when used after nouns or ', None], [None, None, 'な-A